# V7 AlphaForge Training -- Colab**XGBoost ile Trading Model Training**Bu notebook Colab Free/Pro'da V7 trading modelini train eder.GPU: Tesla T4 (16GB VRAM) -- XGBoost icin fazlasiyla yeterli.

In [ ]:
# Colab GPU kontrolimport torchprint(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK'}")if torch.cuda.is_available():    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")from google.colab import drivedrive.mount('/content/drive')print('Drive mount edildi')

In [ ]:
!pip install -q numpy pandas scipy scikit-learn xgboost pyarrow matplotlib seaborn tqdm numba requestsprint('Paketler kuruldu')

In [ ]:
MODE = 'SCALP'SYMBOLS = 'BTCUSDT,ETHUSDT,SOLUSDT,BNBUSDT,ADAUSDT'N_BARS = 3000FOLDS = 6OUTPUT_DIR = '/content/drive/MyDrive/alphaforge_results'print(f'Mode: {MODE}')

In [ ]:
import requests, pandas as pd, numpy as npfrom datetime import datetimeimport timedef fetch_binance_klines(symbol, interval, limit=1000):    url = 'https://api.binance.com/api/v3/klines'    all_klines, end_time = [], int(datetime.now().timestamp() * 1000)    while len(all_klines) < limit:        resp = requests.get(url, params={'symbol':symbol,'interval':interval,'limit':1000,'endTime':end_time}, timeout=30)        if resp.status_code != 200: break        klines = resp.json()        if not klines: break        all_klines = klines + all_klines        if len(klines) < 1000: break        end_time = klines[0][0] - 1; time.sleep(0.1)    df = pd.DataFrame(all_klines[-limit:], columns=['timestamp','open','high','low','close','volume','close_time','quote_vol','trade_count','taker_buy_vol','taker_buy_quote_vol','ignore'])    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')    for c in ['open','high','low','close','volume']: df[c] = df[c].astype(float)    return dfinterval = '1h' if MODE == 'SCALP' else '4h'symbol_list = [s.strip() for s in SYMBOLS.split(',')]all_data = {}for sym in symbol_list:    print(f'{sym} ({interval})...', end=' ')    df = fetch_binance_klines(sym, interval, limit=N_BARS)    all_data[sym] = df; print(f'{len(df)} bar')print(f'Veri: {len(symbol_list)} symbol')

In [ ]:
def compute_features(df):    c, h, l, v = df['close'].values, df['high'].values, df['low'].values, df['volume'].values    feat = {}    for p in [1,2,3,6,12,24]:        a = np.full(len(c), np.nan)        if p < len(c): a[p:] = c[p:] / c[:-p] - 1        feat[f'ret_{p}'] = a    feat['log_ret'] = np.full(len(c), np.nan); feat['log_ret'][1:] = np.log(c[1:]/c[:-1])    ret = c[1:]/c[:-1]-1    for p in [6,12,24]:        a = np.full(len(c), np.nan)        if p+1 < len(c): a[p+1:] = pd.Series(ret).rolling(p).std().values[p:]        feat[f'vol_{p}'] = a    tr = np.maximum(h[1:]-l[1:], np.abs(h[1:]-c[:-1]), np.abs(l[1:]-c[:-1]))    feat['atr'] = np.full(len(c), np.nan); feat['atr'][14:] = pd.Series(np.concatenate([[0],tr])).rolling(14).mean().values[14:]    delta = np.diff(c); gain, loss = np.where(delta>0,delta,0), np.where(delta<0,-delta,0)    ag = pd.Series(gain).rolling(14).mean().values; al = pd.Series(loss).rolling(14).mean().values    feat['rsi'] = np.concatenate([[np.nan], 100-100/(1+ag/np.maximum(al,1e-10))])    sma = pd.Series(c).rolling(20).mean().values; std = pd.Series(c).rolling(20).std().values    feat['bb_z'] = (c-sma)/np.maximum(std,1e-10)    feat['vol_z'] = (v-pd.Series(v).rolling(20).mean())/np.maximum(pd.Series(v).rolling(20).std(),1e-10)    for p in [20,50]:        ma = pd.Series(c).rolling(p).mean().values        feat[f'ma_{p}'] = (c-ma)/np.maximum(ma,1e-10)    feat['close_pos'] = (c-l)/np.maximum(h-l,1e-10)    return pd.DataFrame(feat).ffill().bfill().fillna(0)all_X, all_y = [], []for sym in symbol_list:    df = all_data[sym].copy()    feats = compute_features(df)    fwd = df['close'].shift(-12) / df['close'] - 1    y = np.full(len(df), 2)    y[fwd > 0.003] = 0; y[fwd < -0.003] = 1    valid = ~(fwd.isna() | pd.DataFrame(feats).isna().any(axis=1))    all_X.append(feats[valid].values); all_y.append(y[valid])X = np.vstack(all_X).astype(np.float64); y = np.concatenate(all_y)print(f'Feature matrix: {X.shape}')print(f'LONG={np.sum(y==0)} SHORT={np.sum(y==1)} NO_TRADE={np.sum(y==2)}')

In [ ]:
import xgboost as xgbfrom sklearn.metrics import accuracy_scoreimport timeparams = {'max_depth':6,'learning_rate':0.05,'subsample':0.8,'colsample_bytree':0.8,          'reg_alpha':0.1,'reg_lambda':1.0,'objective':'multi:softprob','num_class':3,          'eval_metric':'mlogloss','random_state':42,'tree_method':'hist'}if torch.cuda.is_available(): params['device']='cuda'; print('GPU mode: cuda')fold_size, fold_results, t0 = len(X)//FOLDS, [], time.time()for fold in range(FOLDS):    vs, ve = fold*fold_size, (fold+1)*fold_size if fold < FOLDS-1 else len(X)    tr, vi = np.arange(0, vs), np.arange(vs, ve)    if len(tr) < 100: continue    dtrain = xgb.DMatrix(X[tr], label=y[tr]); dval = xgb.DMatrix(X[vi], label=y[vi])    model = xgb.train(params, dtrain, num_boost_round=500, evals=[(dtrain,'train'),(dval,'val')], early_stopping_rounds=30, verbose_eval=False)    pred = model.predict(dval).argmax(axis=1); acc = accuracy_score(y[vi], pred)    tacc = accuracy_score(y[tr], model.predict(dtrain).argmax(axis=1))    fold_results.append({'fold':fold+1,'train':len(tr),'val':len(vi),'train_acc':round(tacc,4),'val_acc':round(acc,4),'active':int(np.sum(pred!=2))})    print(f'Fold {fold+1}/{FOLDS}: train={len(tr)} val={len(vi)} acc={acc:.4f} active={int(np.sum(pred!=2))}')train_time=time.time()-t0; val_accs=[r['val_acc']for r in fold_results]final_acc=float(np.mean(val_accs)); overfit=float(np.mean([r['train_acc']for r in fold_results])-final_acc)print('='*55); print(f'ACCURACY (OOS): {final_acc:.4f}'); print(f'OVERFIT GAP: {overfit:.4f}')print(f'SAMPLES: {len(X)} FEATURES: {X.shape[1]} TIME: {train_time:.1f}s'); print('='*55)

In [ ]:
import os, jsonfrom datetime import datetimefinal = xgb.train(params, xgb.DMatrix(X, label=y), num_boost_round=500)ts = datetime.now().strftime('%Y%m%d_%H%M')os.makedirs(OUTPUT_DIR, exist_ok=True)model_path = f'{OUTPUT_DIR}/v7_{MODE.lower()}_xgb_{ts}.json'final.save_model(model_path)imp = final.get_score(importance_type='gain')sorted_imp = sorted(imp.items(), key=lambda x: -x[1])results = {'mode':MODE,'symbols':SYMBOLS,'folds':FOLDS,'n_features':X.shape[1],'n_samples':len(X),'accuracy':final_acc,'overfit_gap':overfit,'fold_results':fold_results,'top_features':[(n,float(v))for n,v in sorted_imp[:20]],'model_path':model_path}with open(f'{OUTPUT_DIR}/v7_{MODE.lower()}_results_{ts}.json','w') as f: json.dump(results,f,indent=2)print(f'Model: {model_path}')print('Top-10 Features:')for n,v in sorted_imp[:10]: print(f'  {n:25s} {v:.4f}')

In [ ]:
import matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.metrics import confusion_matrixpred_final = final.predict(xgb.DMatrix(X)).argmax(axis=1)cm = confusion_matrix(y, pred_final, labels=[0,1,2])plt.figure(figsize=(7,5)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['LONG','SHORT','NO_TRADE'], yticklabels=['LONG','SHORT','NO_TRADE'])plt.title(f'Confusion Matrix - {MODE}'); plt.ylabel('Gercek'); plt.xlabel('Tahmin'); plt.tight_layout()plt.savefig(f'{OUTPUT_DIR}/v7_{MODE.lower()}_confusion_{ts}.png'); plt.show()if fold_results:    plt.figure(figsize=(9,4))    plt.plot([r['fold']for r in fold_results],[r['val_acc']for r in fold_results],'r-o',label='Val')    plt.plot([r['fold']for r in fold_results],[r['train_acc']for r in fold_results],'b-o',label='Train')    plt.axhline(0.333, color='gray', ls='--', label='Random')    plt.xlabel('Fold'); plt.ylabel('Accuracy'); plt.title(f'Walk-Forward - {MODE}')    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()    plt.savefig(f'{OUTPUT_DIR}/v7_{MODE.lower()}_wfv_{ts}.png'); plt.show()print(f'Grafikler: {OUTPUT_DIR}/')

## Notlar- Colab Free: ~12h oturum, T4 GPU, inaktifken kesilir- Colab Pro ($10/ay): uzun oturum, oncelikli GPU- 60 symbol: 5 symbol batch halinde 12x calistir- Feature engineering CPU-bound, training <5GB VRAM- Sonuclar Drive'da alphaforge_results/ klasorunde